# 🎓 Taller NLP — VADER + pysentimiento desde SQL Server

| Parte | Modelo        | Idioma     | Fuente SQL         | Emojis |
| ----- | ------------- | ---------- | ------------------ | ------ |
| **1** | NLTK VADER    | 🇺🇸 Inglés  | `amazon_reviews`   | ✅     |
| **2** | pysentimiento | 🇪🇨 Español | `tweets_valbonesi` | ✅     |

**Pipeline:**

```
SQL Server → pyodbc → DataFrame → modelo NLP → clasificación → UPDATE → SQL Server
```

> **¿Por qué dos modelos?**  
> VADER fue construido con léxico en **inglés**. En español devuelve scores cercanos a 0.0 porque no reconoce las palabras. pysentimiento es un Transformer (RoBERTa) entrenado con millones de tweets en **español latinoamericano**, incluyendo emojis.


---

## ⚙️ Configuración e instalación


In [3]:
# ── Instalación de librerías (ejecutar solo la primera vez) ──────────────────
# nltk          → librería de NLP con VADER para análisis de sentimientos en inglés
# pysentimiento → modelo Transformer para análisis de sentimientos en español
# wordcloud     → genera nubes de palabras visuales
# sqlalchemy    → permite guardar DataFrames directamente en SQL Server con to_sql()
%pip install nltk pysentimiento wordcloud sqlalchemy

# ── Librerías generales ───────────────────────────────────────────────────────
import pyodbc          # Conexión a bases de datos mediante ODBC (SQL Server)
import pandas as pd    # Manipulación de datos en tablas (DataFrames)
import matplotlib.pyplot as plt  # Creación de gráficos
import matplotlib.patches as mpatches  # Elementos visuales extra para gráficos
import seaborn as sns  # Gráficos estadísticos de alto nivel sobre matplotlib
import re              # Expresiones regulares para limpiar texto
import string          # Conjunto de caracteres de puntuación estándar
from datetime import datetime  # Fecha y hora actual para registrar cuándo se analizó

import nltk
nltk.download('vader_lexicon', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

SW_EN = set(stopwords.words('english'))
SW_ES = set(stopwords.words('spanish'))
PUNTUACION = set(string.punctuation) | {'...', '..', '``', "''", '¿', '¡'}

def tokens_limpios(serie, sw):
    '''tokeniza una serie de textos y filtra stopwords, puntuacion y no-palabras'''
    tokens = word_tokenize(' '.join(serie.astype(str)).lower())
    return [t for t in tokens if t not in sw and t not in PUNTUACION and t.isalpha()]

Note: you may need to restart the kernel to use updated packages.


In [7]:
import urllib.parse
from sqlalchemy import create_engine

CONNECTION_STRING = (
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=36469228A21PB2\MSSQLSERVER01;'
    'DATABASE=taller_nlp;'
    'Trusted_Connection=yes;'
)

def get_conn():
    return pyodbc.connect(CONNECTION_STRING)

_params = urllib.parse.quote_plus(CONNECTION_STRING)
engine = create_engine(f'mssql+pyodbc://?odbc_connect={_params}')

try:
    with get_conn() as conn:
        cursor = conn.cursor()
        print('Conexión exitosa')
        for tabla in ['amazon_reviews', 'tweets_valbonesi']:
            cursor.execute(f'SELECT COUNT(*) FROM dbo.{tabla}')
            n = cursor.fetchone()[0]
            print(f'{tabla}: {n} filas')

except pyodbc.Error as e:
    print(f'ERROR: {e}')
    print('Verifica el server')


Conexión exitosa
amazon_reviews: 40 filas
tweets_valbonesi: 60 filas


---

# PARTE 1 — VADER: reseñas Amazon (inglés + emojis)

### ¿Cómo maneja VADER los emojis?

VADER **sí reconoce emojis** en inglés. Su léxico incluye patrones como:

- `🔥` → +0.4 (intensidad positiva)
- `😡` → -0.7
- `💀` → -0.5
- `⭐⭐⭐⭐⭐` → amplifica la positividad

Sin embargo para texto en **español**, VADER ignora las palabras


### 1.1 — Leer desde SQL Server


In [11]:
SQL_REVIEWS = """
    SELECT
        id,
        product,
        stars,
        author,
        review_date,
        title,
        body,
        -- Concatenar title + body para analzar el texto completo con VADER
        title + ' ' + body AS full_text
    FROM dbo.amazon_reviews
    ORDER BY id;
"""

with get_conn() as conn:
    df_rev = pd.read_sql(SQL_REVIEWS, conn, parse_dates=['review_date'])

    print(f'Cargadas: {len(df_rev)} reseñas desde SQL server')
    print(f'Estrellas promedio: {df_rev.stars.mean():.1f} / 5')
    print()

    df_rev[['id','product','stars','author','full_text']].head(4)

Cargadas: 40 reseñas desde SQL server
Estrellas promedio: 3.1 / 5



C:\Users\ANALISIS DE DATOS\AppData\Local\Temp\ipykernel_4292\3134531532.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_rev = pd.read_sql(SQL_REVIEWS, conn, parse_dates=['review_date'])


### 1.2 — Análisis VADER (con emojis)


In [19]:
columnas_vader = [col for col in df_rev.columns if col.startswith('vader_')]
df_rev = df_rev.drop(columns=columnas_vader)

df_rev['full_text'] = df_rev['title'] + ' ' + df_rev['body']

sia = SentimentIntensityAnalyzer()

def analizar_sentimiento(texto):
    scores = sia.polarity_scores(texto)
    
    if scores['compound'] >= 0.05:
        scores['label'] = 'Positive'
    elif scores['compound'] <= -0.05:
        scores['label'] = 'Negative'
    else:
        scores['label'] = 'Neutral'
    
    return scores

resultados = df_rev['full_text'].apply(analizar_sentimiento)

resultados = resultados.apply(pd.Series)

df_rev = pd.concat([df_rev, resultados.add_prefix('vader_')], axis=1)

df_rev['vader_label'].value_counts().rename('resenas')

vader_label
Positive    28
Negative    12
Name: resenas, dtype: int64

### 1.3 — Guardar resultados en SQL Server


In [22]:
df_vader_export = df_rev[['id','product','stars','title','body','vader_compound','vader_neg','vader_neu', 'vader_pos','vader_label']].copy()

df_vader_export['analyzed_at'] = datetime.now()

df_vader_export.to_sql('resultados_vader', con=engine, schema='dbo', if_exists='replace', index=False)

print(f'{len(df_vader_export)} registros guardados en dbo.resultados_vader')

40 registros guardados en dbo.resultados_vader


---

# PARTE 2 — pysentimiento: tweets Valbonesi (español + emojis)

### ¿Por qué pysentimiento?

```python
# VADER en español: FALLA
sia.polarity_scores('Qué vergüenza académica!! 😡')
# → {'neg': 0.0, 'neu': 0.516, 'pos': 0.0, 'compound': 0.0}  ← NO entiende 'vergüenza'

# pysentimiento en español: CORRECTO
analyzer_es.predict('Qué vergüenza académica!! 😡')
# → SentimentOutput(output='NEG', probas={'NEG': 0.91, 'NEU': 0.06, 'POS': 0.03})
```

Pysentimiento también procesa **emojis nativamente** porque fue entrenado con tweets reales que los incluían.


In [17]:
from pysentimiento import create_analyzer

analyzer_es = create_analyzer(task='sentiment', lang='es')
print('pysentimiento listo')

ejemplo = 'Qué vergüenza académica!! 😡'

v = sia.polarity_scores(ejemplo)['compound']

p = analyzer_es.predict(ejemplo).output

print(f'texto: {ejemplo}')
print(f'VADER: compund {v}')

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

pysentimiento listo
texto: Qué vergüenza académica!! 😡
VADER: compund 0.0


### 2.1 — Leer tweets desde SQL Server


In [27]:
SQL_TWEETS = """
    select
        id,
        usuario,
        tweet_date,
        likes,
        retweets,
        texto
    from dbo.tweets_valbonesi
    order by tweet_date, id
"""

with get_conn() as conn:
    df_tw = pd.read_sql(SQL_TWEETS, conn, parse_dates=['tweet_date'])

print(f'Cargados {len(df_tw)} cargados')


Cargados 60 cargados


C:\Users\ANALISIS DE DATOS\AppData\Local\Temp\ipykernel_4292\330087593.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_tw = pd.read_sql(SQL_TWEETS, conn, parse_dates=['tweet_date'])


### 2.2 — Análisis pysentimiento


In [28]:
def classify_pysentimiento(texto):
    r = analyzer_es.predict(str(texto))
    return {'label': r.output,
            'prob_pos': round(r.probas.get('POS', 0), 4),
            'prob_neg': round(r.probas.get('NEG', 0), 4),
            'prob_neu': round(r.probas.get('NEU', 0), 4)}

print('Analizando tweets...')

py_results = df_tw['texto'].apply(classify_pysentimiento).apply(pd.Series)

df_tw = pd.concat([df_tw, py_results.add_prefix('py_')], axis=1)

df_tw['sentimiento'] = df_tw['py_label'].map({'POS': 'Positivo',
                                              'NEG': 'Negativo',
                                              'NEU': 'Neutral'})

df_tw['sentimiento'].value_counts().rename('resenas')

Analizando tweets...


sentimiento
Negativo    29
Neutral     19
Positivo    12
Name: resenas, dtype: int64

### 2.3 — Guardar resultados en SQL Server


In [29]:
df_pysentmiento_export = df_tw[['id','usuario','tweet_date','likes','retweets','py_label','py_prob_pos','py_prob_neg','py_prob_neu','sentimiento']].copy()

df_pysentmiento_export['analyzed_at'] = datetime.now()

df_pysentmiento_export.to_sql('resultados_pysentimiento', con=engine, schema='dbo', if_exists='replace', index=False)

print(f'{len(df_pysentmiento_export)} registros guardados con éxito')

60 registros guardados con éxito
